<a href="https://colab.research.google.com/github/Marfall/RecommendationSystem-Otus-10/blob/main/RecommendationSystem_Otus_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание №10: Рекомендательная система на Amazon Digital Music

**Цель:** Построить рекомендательную систему на реальных данных Amazon и сравнить несколько подходов.

**Датасет:** Amazon Digital Music (5-core subset). Отзывы на цифровую музыку с рейтингами и временными метками.

**План:**
1. Импорт библиотек.
2. Загрузка данных.
3. EDA: распределения, уникальные юзеры и товары.
4. Leave-one-out разбиение.
5. Baseline «популярные товары».
6. Item-based collaborative filtering.
7. SVD-модель.
8. Оценка: HR@10, MRR@10, NDCG@10, coverage.
9. Выводы.

## 1. Импорт библиотек

Загружаются pandas, numpy, scipy для разреженных матриц, sklearn для SVD, matplotlib/seaborn для графиков.

In [ ]:
# 2: Импорт библиотек
import os, gzip, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

sns.set_style('whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

np.random.seed(42)
print('Библиотеки импортированы.')

## 3. Загрузка данных

Скачиваем Digital Music 5-core (~30 МБ в архиве). Формат — JSON Lines: одна строка = один отзыв.

In [ ]:
# 4: Загрузка данных
import urllib.request

URL = ('https://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/'
       'Digital_Music_5.json.gz')
LOCAL_PATH = '/content/digital_music.json.gz'

if not os.path.exists(LOCAL_PATH):
    print('Скачивание...')
    urllib.request.urlretrieve(URL, LOCAL_PATH)
    print(f'Скачано: {os.path.getsize(LOCAL_PATH) / 1e6:.1f} МБ')
else:
    print('Файл уже скачан.')

# Читаем JSON lines
records = []
with gzip.open(LOCAL_PATH, 'rt', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'\nРазмер: {df.shape}')
print(f'Колонки: {df.columns.tolist()}')
print(df.head())

## 5. Базовый EDA

Смотрим распределение рейтингов, количество уникальных юзеров и товаров, активность юзеров.

In [ ]:
# 6: EDA
print(f'Всего отзывов: {len(df)}')
print(f'Уникальных юзеров: {df["reviewerID"].nunique()}')
print(f'Уникальных товаров: {df["asin"].nunique()}')

# Распределение рейтингов
rating_dist = df['overall'].value_counts().sort_index()
print('\n=== Распределение рейтингов ===')
rating_df = pd.DataFrame({
    'Рейтинг': rating_dist.index,
    'Кол-во': rating_dist.values,
    'Доля, %': (rating_dist.values / len(df) * 100).round(2)
})
print(rating_df.to_string(index=False))

# Активность юзеров
user_counts = df.groupby('reviewerID').size()
print(f'\nОтзывов на юзера — медиана: {user_counts.median():.0f}, '
      f'среднее: {user_counts.mean():.1f}, максимум: {user_counts.max()}')

# Активность товаров
item_counts = df.groupby('asin').size()
print(f'Отзывов на товар — медиана: {item_counts.median():.0f}, '
      f'среднее: {item_counts.mean():.1f}, максимум: {item_counts.max()}')

# Графики
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(x=rating_df['Рейтинг'], y=rating_df['Кол-во'], ax=axes[0])
axes[0].set_title('Распределение рейтингов')
axes[0].set_xlabel('Рейтинг')
axes[0].set_ylabel('Количество')

sns.histplot(user_counts, bins=50, ax=axes[1])
axes[1].set_title('Сколько отзывов у одного юзера')
axes[1].set_xlabel('Число отзывов')
axes[1].set_xlim(0, 50)

plt.tight_layout()
plt.show()
plt.close('all')

# Сводка
eda_summary = pd.DataFrame({
    'Метрика': ['Отзывов', 'Юзеров', 'Товаров', 'Средний рейтинг',
                'Медиана отзывов/юзер', 'Медиана отзывов/товар'],
    'Значение': [len(df), df['reviewerID'].nunique(), df['asin'].nunique(),
                 round(df['overall'].mean(), 2),
                 int(user_counts.median()), int(item_counts.median())]
})
print('\n=== Сводка EDA ===')
print(eda_summary.to_string(index=False))

## 7. Leave-one-out разбиение

Для каждого юзера последний по времени отзыв уходит в тест, остальные — в train. Юзеры с одним отзывом отбрасываются (иначе в train для них ничего не останется).

In [ ]:
# 8: Leave-one-out
df_sorted = df.sort_values(['reviewerID', 'unixReviewTime'])

train_rows = []
test_rows = []

for uid, group in tqdm(df_sorted.groupby('reviewerID'), desc='Leave-one-out'):
    if len(group) < 2:
        continue
    train_rows.append(group.iloc[:-1])
    test_rows.append(group.iloc[-1:])

train_df = pd.concat(train_rows, ignore_index=True)
test_df = pd.concat(test_rows, ignore_index=True)

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(f'Юзеров в train: {train_df["reviewerID"].nunique()}')
print(f'Юзеров в test: {test_df["reviewerID"].nunique()}')

# Проверяем, что у всех тестовых юзеров есть история в train
common_users = set(train_df['reviewerID']) & set(test_df['reviewerID'])
print(f'Пересечение юзеров train/test: {len(common_users)}')

# Ограничение для скорости
MAX_USERS = 5000
if test_df['reviewerID'].nunique() > MAX_USERS:
    keep_users = set(test_df['reviewerID'].unique()[:MAX_USERS])
    train_df = train_df[train_df['reviewerID'].isin(keep_users)].copy()
    test_df = test_df[test_df['reviewerID'].isin(keep_users)].copy()
    print(f'\nОграничили до {MAX_USERS} юзеров для скорости.')
    print(f'Train: {len(train_df)} | Test: {len(test_df)}')

## 9. Построение user-item матрицы

Индексируем юзеров и товары, строим разреженную матрицу взаимодействий. Значения — рейтинги.

In [ ]:
# 10: User-item матрица
user_ids = train_df['reviewerID'].unique().tolist()
item_ids = train_df['asin'].unique().tolist()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}

rows = train_df['reviewerID'].map(user_to_idx).values
cols = train_df['asin'].map(item_to_idx).values
vals = train_df['overall'].values.astype(np.float32)

R = sp.csr_matrix((vals, (rows, cols)),
                  shape=(len(user_ids), len(item_ids)))
print(f'Матрица: {R.shape}, ненулевых: {R.nnz}')
print(f'Плотность: {R.nnz / (R.shape[0] * R.shape[1]) * 100:.4f}%')

# Индекс товаров по имени для быстрого поиска
idx_to_item = {i: it for it, i in item_to_idx.items()}
item_to_name = dict(zip(df['asin'], df['title'].fillna('').str[:60]))

## 11. Baseline: популярные товары

Рекомендуем всем юзерам одни и те же топ-10 самых популярных товаров из train. Юзеры, у которых эти товары уже есть в истории, эти товары не получают (хотя для базовой модели это не критично — метрики окажутся низкими всё равно).

In [ ]:
# 12: Baseline — популярные товары
TOP_K = 10

popular_items = (train_df.groupby('asin').size()
                 .sort_values(ascending=False)
                 .head(TOP_K).index.tolist())
popular_indices = [item_to_idx[it] for it in popular_items]

print('=== Топ-10 популярных товаров ===')
pop_df = pd.DataFrame({
    'asin': popular_items,
    'Название': [item_to_name.get(it, '')[:50] for it in popular_items],
    'Покупок в train': [int(train_df[train_df['asin'] == it].shape[0])
                        for it in popular_items]
})
print(pop_df.to_string(index=False))

# Функция оценки метрик
def evaluate_model(recommend_fn, test_df, train_df, K=10):
    """recommend_fn(uid) -> список из K индексов товаров."""
    hrs, mrrs, ndcgs = [], [], []
    all_recommended = set()

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Оценка'):
        uid = row['reviewerID']
        true_item = row['asin']
        if true_item not in item_to_idx:
            continue
        true_idx = item_to_idx[true_item]

        recs = recommend_fn(uid)
        all_recommended.update(recs)

        if true_idx in recs:
            rank = recs.index(true_idx) + 1
            hrs.append(1)
            mrrs.append(1 / rank)
            ndcgs.append(1 / np.log2(rank + 1))
        else:
            hrs.append(0)
            mrrs.append(0)
            ndcgs.append(0)

    coverage = len(all_recommended) / len(item_ids)
    return {
        'HR@10': np.mean(hrs),
        'MRR@10': np.mean(mrrs),
        'NDCG@10': np.mean(ndcgs),
        'Coverage': coverage
    }

def recommend_popular(uid, K=TOP_K):
    return popular_indices[:K]

baseline_metrics = evaluate_model(recommend_popular, test_df, train_df)
print('\n=== Baseline метрики ===')
for k, v in baseline_metrics.items():
    print(f'{k}: {v:.4f}')

## 13. Item-based Collaborative Filtering

Считаем косинусную близость между товарами через матрицу user-item. Для юзера берём его историю из train, находим товары, наиболее близкие к тому, что он уже слушал, и рекомендуем топ-10.

In [ ]:
# 14: Item-based CF
print('Считаем item-item близость...')
item_sim = cosine_similarity(R.T)  # (n_items, n_items)
np.fill_diagonal(item_sim, 0)
print(f'Матрица близости: {item_sim.shape}')

def recommend_item_cf(uid, K=TOP_K):
    if uid not in user_to_idx:
        return recommend_popular(uid, K)
    u_idx = user_to_idx[uid]
    user_items = R[u_idx].toarray().flatten()
    interacted = np.where(user_items > 0)[0]

    if len(interacted) == 0:
        return recommend_popular(uid, K)

    # Считаем скоринг: сумма близостей к уже просмотренным
    scores = item_sim[interacted].sum(axis=0)
    # Убираем то, что уже есть у юзера
    scores[interacted] = -np.inf
    top_idx = np.argsort(scores)[-K:][::-1]
    return top_idx.tolist()

itemcf_metrics = evaluate_model(recommend_item_cf, test_df, train_df)
print('\n=== Item-based CF метрики ===')
for k, v in itemcf_metrics.items():
    print(f'{k}: {v:.4f}')

## 15. SVD

Разложение user-item матрицы через TruncatedSVD на 50 компонент. Каждому юзеру и товару сопоставляется вектор в скрытом пространстве. Рекомендуем товары с наибольшим скалярным произведением векторов.

In [ ]:
# 16: SVD
N_COMPONENTS = 50
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
U = svd.fit_transform(R)
Sigma = np.diag(svd.singular_values_)
Vt = svd.components_

# Реконструированные оценки
R_hat = U @ Sigma @ Vt
print(f'SVD: {N_COMPONENTS} компонент, объяснённая дисперсия: '
      f'{svd.explained_variance_ratio_.sum():.3f}')

def recommend_svd(uid, K=TOP_K):
    if uid not in user_to_idx:
        return recommend_popular(uid, K)
    u_idx = user_to_idx[uid]
    scores = R_hat[u_idx].copy()
    # Убираем уже просмотренное
    interacted = np.where(R[u_idx].toarray().flatten() > 0)[0]
    scores[interacted] = -np.inf
    top_idx = np.argsort(scores)[-K:][::-1]
    return top_idx.tolist()

svd_metrics = evaluate_model(recommend_svd, test_df, train_df)
print('\n=== SVD метрики ===')
for k, v in svd_metrics.items():
    print(f'{k}: {v:.4f}')

## 17. Сравнение моделей

Сводная таблица метрик и визуализация.

In [ ]:
# 18: Сравнение моделей
comparison = pd.DataFrame([
    {'Модель': 'Popularity', **baseline_metrics},
    {'Модель': 'Item-based CF', **itemcf_metrics},
    {'Модель': 'SVD', **svd_metrics},
])
print('=== Сравнение моделей ===')
print(comparison.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best_model = comparison.loc[comparison['HR@10'].idxmax(), 'Модель']
print(f'\nЛучшая модель по HR@10: {best_model}')

# График
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison))
width = 0.2
metrics = ['HR@10', 'MRR@10', 'NDCG@10', 'Coverage']

for i, m in enumerate(metrics):
    bars = ax.bar(x + (i - 1.5) * width, comparison[m], width, label=m)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(comparison['Модель'])
ax.set_ylabel('Значение')
ax.set_title('Метрики рекомендательных моделей')
ax.legend()
plt.tight_layout()
plt.show()
plt.close('all')

## 19. Выводы

**Заполнить после запуска:**

- Юзеров: `[N_USERS]`
- Товаров: `[N_ITEMS]`
- Popularity: HR=`[POP_HR]`, MRR=`[POP_MRR]`, NDCG=`[POP_NDCG]`, Coverage=`[POP_COV]`
- Item-based CF: HR=`[CF_HR]`, MRR=`[CF_MRR]`, NDCG=`[CF_NDCG]`, Coverage=`[CF_COV]`
- SVD: HR=`[SVD_HR]`, MRR=`[SVD_MRR]`, NDCG=`[SVD_NDCG]`, Coverage=`[SVD_COV]`
- Лучшая модель: `[BEST_MODEL]`

**Что получилось:**

[INTERPRETATION]

In [ ]:
# 20: Финальная сводка
final_summary = pd.DataFrame({
    'Модель': comparison['Модель'],
    'HR@10': comparison['HR@10'].round(4),
    'MRR@10': comparison['MRR@10'].round(4),
    'NDCG@10': comparison['NDCG@10'].round(4),
    'Coverage': comparison['Coverage'].round(4),
})
print('=== Финальная сводка ===')
print(final_summary.to_string(index=False))

# Переменные для выводов
N_USERS = test_df['reviewerID'].nunique()
N_ITEMS = len(item_ids)
POP_HR = baseline_metrics['HR@10']
CF_HR = itemcf_metrics['HR@10']
SVD_HR = svd_metrics['HR@10']